# Task 02 — Customer Churn Baseline Notebook

This notebook documents a reproducible baseline and responsible evaluation for the supplied 12-row customer-churn dataset. The tiny sample means results are diagnostic only, not production evidence.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score

df = pd.read_csv('customer-churn-training.csv')
df

## 1. Data quality checks

Check shape, schema, missing values, duplicates, and target balance before modeling.

In [ ]:
print('Shape:', df.shape)
print('\nDtypes:\n', df.dtypes)
print('\nMissing values:\n', df.isna().sum())
print('\nDuplicate rows:', df.duplicated().sum())
print('\nTarget balance:\n', df['churned'].value_counts())
print('\nPlan type by churn:\n', pd.crosstab(df['plan_type'], df['churned']))

## 2. Non-ML operational baseline

Baseline rule: flag customers with `plan_type == 'Basic'`. This is a transparent comparator, not a claim that plan type should determine real customer treatment.

In [ ]:
y = df['churned']
baseline_pred = (df['plan_type'] == 'Basic').astype(int)

print('Accuracy:', accuracy_score(y, baseline_pred))
print('Precision:', precision_score(y, baseline_pred, zero_division=0))
print('Recall:', recall_score(y, baseline_pred, zero_division=0))
print('F1:', f1_score(y, baseline_pred, zero_division=0))
print('Confusion matrix:\n', confusion_matrix(y, baseline_pred))

## 3. Logistic-regression baseline

Use numeric features plus one-hot encoded `plan_type`. Exclude `customer_id`. Leave-one-out cross-validation is used because a conventional train/test split would be extremely unstable with only 12 rows.

In [ ]:
features = ['tenure_months', 'support_tickets', 'monthly_spend_inr', 'last_login_days', 'plan_type']
X = df[features]

pre = ColumnTransformer([
    ('num', StandardScaler(), ['tenure_months', 'support_tickets', 'monthly_spend_inr', 'last_login_days']),
    ('cat', OneHotEncoder(handle_unknown='ignore'), ['plan_type'])
])

model = Pipeline([
    ('preprocess', pre),
    ('classifier', LogisticRegression(max_iter=1000))
])

loo = LeaveOneOut()
pred = cross_val_predict(model, X, y, cv=loo, method='predict')
proba = cross_val_predict(model, X, y, cv=loo, method='predict_proba')[:, 1]

print('Accuracy:', accuracy_score(y, pred))
print('Precision:', precision_score(y, pred, zero_division=0))
print('Recall:', recall_score(y, pred, zero_division=0))
print('F1:', f1_score(y, pred, zero_division=0))
print('ROC-AUC:', roc_auc_score(y, proba))
print('Confusion matrix:\n', confusion_matrix(y, pred))

## 4. Responsible interpretation

Both the simple baseline and logistic-regression baseline can appear perfect on this dataset. That result must **not** be presented as production accuracy. The dataset has only 12 observations, and `plan_type` perfectly separates the observed labels (all Basic customers churned; all Standard/Pro customers did not). There is no time field, so temporal generalization and leakage cannot be established.

Before deployment: collect a larger time-stamped dataset, define the prediction/action window, document provenance and permission, establish business costs for false positives/negatives, and validate on a future holdout period.